# Problem Statement

## **Business Context**

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.


## **Objective**

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## **Data Description**

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


In [2]:
from huggingface_hub import notebook_login
notebook_login()

# Model Building

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

PROJECT_DIR = "/content/drive/MyDrive/Colab_Notebooks/tourism_project"
print(os.listdir(PROJECT_DIR))

Mounted at /content/drive
['data', 'deployment', 'model_building', 'Learner_Template_Notebook_AML_and_MLOps_Project.ipynb']


In [8]:
# Create a folder for storing the model building files
os.makedirs("tourism_project/model_building", exist_ok=True)

In [3]:
import os

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Colab_Notebooks/tourism_project"
RUNTIME_PROJECT_DIR = "/content/tourism_project"

print("Drive project exists:", os.path.exists(DRIVE_PROJECT_DIR))
print("Runtime project exists:", os.path.exists(RUNTIME_PROJECT_DIR))

print("\nRuntime folder contents (if exists):")
if os.path.exists(RUNTIME_PROJECT_DIR):
    print(os.listdir(RUNTIME_PROJECT_DIR))

print("\nDrive folder contents:")
print(os.listdir(DRIVE_PROJECT_DIR))

Drive project exists: True
Runtime project exists: False

Runtime folder contents (if exists):

Drive folder contents:
['data', 'deployment', 'model_building', 'Learner_Template_Notebook_AML_and_MLOps_Project.ipynb']


In [4]:
import os

DRIVE_CSV = "/content/drive/MyDrive/Colab_Notebooks/tourism_project/data/tourism.csv"
print("Drive tourism.csv exists?", os.path.exists(DRIVE_CSV))

Drive tourism.csv exists? True


In [5]:
RUNTIME_CSV = "/content/tourism_project/data/tourism.csv"
print("Runtime tourism.csv exists?", os.path.exists(RUNTIME_CSV))

Runtime tourism.csv exists? False


In [6]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Colab_Notebooks/tourism_project"
MODEL_BUILDING_DIR = os.path.join(PROJECT_DIR, "model_building")
DATA_DIR = os.path.join(PROJECT_DIR, "data")

print("PROJECT_DIR:", PROJECT_DIR)
print("MODEL_BUILDING_DIR exists?", os.path.exists(MODEL_BUILDING_DIR))
print("DATA_DIR exists?", os.path.exists(DATA_DIR))
print("DATA_DIR contents:", os.listdir(DATA_DIR))

PROJECT_DIR: /content/drive/MyDrive/Colab_Notebooks/tourism_project
MODEL_BUILDING_DIR exists? True
DATA_DIR exists? True
DATA_DIR contents: ['tourism.csv']


## Data Registration

In [9]:
os.makedirs("tourism_project/data", exist_ok=True)

Once the **data** folder created after executing the above cell, please upload the **tourism.csv** in to the folder

In [14]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/tourism_project/model_building/data_register.py
import os
from huggingface_hub import HfApi, create_repo

def register_data():
    """
    Registers raw data on Hugging Face Dataset Hub.
    Expects:
      HF_DATASET_REPO : e.g., "username/tourism-package-data"
      LOCAL_CSV_PATH  : e.g., "/content/drive/MyDrive/Colab_Notebooks/tourism_project/data/tourism.csv"
      HF_RAW_FILENAME : optional, default "tourism.csv"
    """
    hf_repo_id = os.environ.get("HF_DATASET_REPO")
    local_csv_path = os.environ.get("LOCAL_CSV_PATH")
    hf_filename = os.environ.get("HF_RAW_FILENAME", "tourism.csv")

    if not hf_repo_id:
        raise ValueError("HF_DATASET_REPO is missing. Example: 'username/tourism-package-data'")
    if not local_csv_path:
        raise ValueError("LOCAL_CSV_PATH is missing. Example: '/content/drive/MyDrive/Colab_Notebooks/tourism_project/data/tourism.csv'")
    if not os.path.exists(local_csv_path):
        raise FileNotFoundError(f"CSV file not found at: {local_csv_path}")

    # Create dataset repo (if it already exists, exist_ok=True prevents failure)
    create_repo(repo_id=hf_repo_id, repo_type="dataset", exist_ok=True)

    # Upload raw CSV to HF dataset repo
    api = HfApi()
    api.upload_file(
        path_or_fileobj=local_csv_path,
        path_in_repo=hf_filename,
        repo_id=hf_repo_id,
        repo_type="dataset"
    )

    print(f" Data registered successfully to HF dataset repo: {hf_repo_id} (file: {hf_filename})")

if __name__ == "__main__":
    register_data()

Writing /content/drive/MyDrive/Colab_Notebooks/tourism_project/model_building/data_register.py


## Data Preparation

In [15]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/tourism_project/model_building/prep.py
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi, create_repo

def upload_to_hf_dataset(repo_id: str, local_path: str, path_in_repo: str) -> None:
    create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
    api = HfApi()
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=path_in_repo,
        repo_id=repo_id,
        repo_type="dataset"
    )

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Drop obvious non-predictive / index-like columns if present
    drop_cols = [c for c in ["CustomerID", "Unnamed: 0"] if c in df.columns]
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

    # Basic string cleanup for object columns (safe, non-destructive)
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()

    return df

def main():
    # Required env vars (friendly for CI/CD)
    hf_repo_id = os.environ.get("HF_DATASET_REPO")
    local_csv_path = os.environ.get("LOCAL_CSV_PATH")  # repo path in CI/CD will be like: "data/tourism.csv"

    if not hf_repo_id:
        raise ValueError("Missing HF_DATASET_REPO. Example: 'abhisekbasu/tourism-package-data'")
    if not local_csv_path:
        raise ValueError("Missing LOCAL_CSV_PATH. Example: 'data/tourism.csv' or '/content/.../data/tourism.csv'")
    if not os.path.exists(local_csv_path):
        raise FileNotFoundError(f"Raw CSV not found at: {local_csv_path}")

    target_col = os.environ.get("TARGET_COL", "ProdTaken")
    test_size = float(os.environ.get("TEST_SIZE", "0.2"))
    random_state = int(os.environ.get("RANDOM_STATE", "42"))

    # Output paths (local)
    output_dir = os.environ.get("OUTPUT_DIR", "data/processed")
    os.makedirs(output_dir, exist_ok=True)

    # 1) Upload raw data to HF (Option 2 requirement)
    upload_to_hf_dataset(hf_repo_id, local_csv_path, "raw/tourism.csv")
    print("Uploaded raw data to HF:", f"{hf_repo_id}/raw/tourism.csv")

    # 2) Load + clean
    df = pd.read_csv(local_csv_path)
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataset columns: {list(df.columns)}")

    df_clean = clean_data(df)

    # 3) Split (stratified to preserve target distribution)
    y = df_clean[target_col]
    X = df_clean.drop(columns=[target_col])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    train_df = pd.concat([X_train, y_train], axis=1)
    test_df = pd.concat([X_test, y_test], axis=1)

    train_path = os.path.join(output_dir, "train.csv")
    test_path = os.path.join(output_dir, "test.csv")

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Saved locally:", train_path, test_path)

    # 4) Upload train/test back to HF dataset repo
    upload_to_hf_dataset(hf_repo_id, train_path, "processed/train.csv")
    upload_to_hf_dataset(hf_repo_id, test_path, "processed/test.csv")

    print("Uploaded processed splits to HF:",
          f"{hf_repo_id}/processed/train.csv and {hf_repo_id}/processed/test.csv")

if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/Colab_Notebooks/tourism_project/model_building/prep.py


## Model Training and Registration with Experimentation Tracking

####Experimentation & Tracking (Development Environment)

In [26]:
!pip install mlflow==3.0.1 pyngrok==7.2.12 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.1/811.1 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.4 MB/s eta 0:00:00


In [17]:
from pyngrok import ngrok
import subprocess
import mlflow

# Set your auth token here (replace with your actual token)
ngrok.set_auth_token("3AGbB0vih33a0JkrGyzkcQ312WT_2EGJWa4SquLMxCMRSoToi")

# Start MLflow UI on port 5000
process = subprocess.Popen(["mlflow", "ui", "--port", "5000"])

# Create public tunnel
public_url = ngrok.connect(5000).public_url
print("MLflow UI is available at:", public_url)

# Set the tracking URL for MLflow
mlflow.set_tracking_uri(public_url)

MLflow UI is available at: https://companyless-leif-desertedly.ngrok-free.dev


In [18]:
mlflow.set_experiment("MLOps_experiment_test")

with mlflow.start_run(run_name="smoke_test"):
    mlflow.log_param("test_param", 123)
    mlflow.log_metric("test_metric", 0.987)
    print("✅ Logged one test run to MLflow.")

2026/02/27 19:52:15 INFO mlflow.tracking.fluent: Experiment with name 'MLOps_experiment_test' does not exist. Creating a new experiment.


✅ Logged one test run to MLflow.
🏃 View run smoke_test at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/558615749665138867/runs/70b8b7565f234682955135bca8320ea2
🧪 View experiment at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/558615749665138867


In [19]:
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

import mlflow

In [20]:
tourism_path = "/content/drive/MyDrive/Colab_Notebooks/tourism_project/data/tourism.csv"
df = pd.read_csv(tourism_path)

print("Loaded:", df.shape)
print("Columns:", list(df.columns))
df.head()

Loaded: (4128, 21)
Columns: ['Unnamed: 0', 'CustomerID', 'ProdTaken', 'Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome']


,Unnamed: 0,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,...,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,...,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,...,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,...,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,...,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,5,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,...,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


In [21]:
drop_cols = [c for c in ["CustomerID", "Unnamed: 0"] if c in df.columns]
print("Dropping:", drop_cols)

df = df.drop(columns=drop_cols) if drop_cols else df
print("After drop:", df.shape)

Dropping: ['CustomerID', 'Unnamed: 0']
After drop: (4128, 19)


In [22]:
target_col = "ProdTaken"

X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", Xtrain.shape, "Test:", Xtest.shape)
print("Target dist (train):")
print(ytrain.value_counts(normalize=True))

Train: (3302, 18) Test: (826, 18)
Target dist (train):
ProdTaken
0    0.806784
1    0.193216
Name: proportion, dtype: float64


In [23]:
print(Xtrain.dtypes)

Age                         float64
TypeofContact                object
CityTier                      int64
DurationOfPitch             float64
Occupation                   object
Gender                       object
NumberOfPersonVisiting        int64
NumberOfFollowups           float64
ProductPitched               object
PreferredPropertyStar       float64
MaritalStatus                object
NumberOfTrips               float64
Passport                      int64
PitchSatisfactionScore        int64
OwnCar                        int64
NumberOfChildrenVisiting    float64
Designation                  object
MonthlyIncome               float64
dtype: object


In [24]:
numeric_features = [
    "Age",
    "CityTier",
    "DurationOfPitch",
    "NumberOfPersonVisiting",
    "NumberOfFollowups",
    "PreferredPropertyStar",
    "NumberOfTrips",
    "Passport",
    "PitchSatisfactionScore",
    "OwnCar",
    "NumberOfChildrenVisiting",
    "MonthlyIncome"
]

categorical_features = [
    "TypeofContact",
    "Occupation",
    "Gender",
    "ProductPitched",
    "MaritalStatus",
    "Designation"
]

# sanity check (must match Xtrain columns exactly)
print("Missing numeric:", [c for c in numeric_features if c not in Xtrain.columns])
print("Missing categorical:", [c for c in categorical_features if c not in Xtrain.columns])
print("Extra columns not listed:", [c for c in Xtrain.columns if c not in numeric_features + categorical_features])

Missing numeric: []
Missing categorical: []
Extra columns not listed: []


In [25]:
# Preprocessing: scale numeric + one-hot encode categorical
preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown="ignore"), categorical_features)
)

# Handle class imbalance like sample (safe even if balanced)
class_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]
print("scale_pos_weight:", class_weight)

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=class_weight,
    random_state=42,
    eval_metric="logloss"
)

param_grid = {
    "xgbclassifier__n_estimators": [80, 120],
    "xgbclassifier__max_depth": [3, 4],
    "xgbclassifier__colsample_bytree": [0.5, 0.7],
    "xgbclassifier__learning_rate": [0.05, 0.1],
    "xgbclassifier__reg_lambda": [0.5, 1.0],
}

model_pipeline = make_pipeline(preprocessor, xgb_model)

scale_pos_weight: 4.175548589341693


In [26]:
mlflow.set_experiment("MLOps_experiment")

with mlflow.start_run(run_name="tourism_dev_gridsearch"):
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, n_jobs=-1)
    grid_search.fit(Xtrain, ytrain)

    # Log all parameter combinations and their mean test scores (proof for rubric)
    results = grid_search.cv_results_
    for i in range(len(results["params"])):
        param_set = results["params"][i]
        mean_score = results["mean_test_score"][i]
        std_score = results["std_test_score"][i]

        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_test_score", float(mean_score))
            mlflow.log_metric("std_test_score", float(std_score))

    # Log best params
    mlflow.log_params(grid_search.best_params_)

    best_model = grid_search.best_estimator_

    # Evaluate with a fixed threshold like sample
    classification_threshold = 0.45
    y_pred_train_proba = best_model.predict_proba(Xtrain)[:, 1]
    y_pred_train = (y_pred_train_proba >= classification_threshold).astype(int)

    y_pred_test_proba = best_model.predict_proba(Xtest)[:, 1]
    y_pred_test = (y_pred_test_proba >= classification_threshold).astype(int)

    train_report = classification_report(ytrain, y_pred_train, output_dict=True)
    test_report = classification_report(ytest, y_pred_test, output_dict=True)

    mlflow.log_metrics({
        "train_accuracy": train_report["accuracy"],
        "train_precision": train_report["1"]["precision"],
        "train_recall": train_report["1"]["recall"],
        "train_f1": train_report["1"]["f1-score"],
        "test_accuracy": test_report["accuracy"],
        "test_precision": test_report["1"]["precision"],
        "test_recall": test_report["1"]["recall"],
        "test_f1": test_report["1"]["f1-score"],
    })

print("✅ Dev experimentation run logged to MLflow.")
print("Best params:", grid_search.best_params_)

2026/02/27 19:57:13 INFO mlflow.tracking.fluent: Experiment with name 'MLOps_experiment' does not exist. Creating a new experiment.


🏃 View run respected-wren-105 at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/862022315284359310/runs/a122f02acc3a4779a268be36c578653b
🧪 View experiment at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/862022315284359310
🏃 View run merciful-fish-834 at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/862022315284359310/runs/2cb1aecb06134ec4aa8b12d8820cc1ba
🧪 View experiment at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/862022315284359310
🏃 View run stylish-crab-88 at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/862022315284359310/runs/7fdc8b43a2a24ca1a83cf06e49d609f9
🧪 View experiment at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/862022315284359310
🏃 View run masked-sheep-583 at: https://companyless-leif-desertedly.ngrok-free.dev/#/experiments/862022315284359310/runs/f6639936ad484db89f0bd5f2403c3646
🧪 View experiment at: https://companyless-leif-desertedly.ngrok-

#### Experimentation & Tracking (Production Environment)

In [27]:
import os

# -------------------------
# Hugging Face repos
# -------------------------
os.environ["HF_DATASET_REPO"] = "abhisekbasu/tourism-project-data"
os.environ["HF_MODEL_REPO"] = "abhisekbasu/tourism-project-model"

# -------------------------
# Target column
# -------------------------
os.environ["TARGET_COL"] = "ProdTaken"

# -------------------------
# MLflow tracking (Dev environment)
# -------------------------
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT"] = "MLOps_experiment"

# -------------------------
# Optional threshold override
# -------------------------
os.environ["CLASSIFICATION_THRESHOLD"] = "0.45"

print("✅ Environment variables set successfully.")

✅ Environment variables set successfully.


In [28]:
print("HF_DATASET_REPO:", os.environ.get("HF_DATASET_REPO"))
print("HF_MODEL_REPO:", os.environ.get("HF_MODEL_REPO"))
print("TARGET_COL:", os.environ.get("TARGET_COL"))
print("MLFLOW_TRACKING_URI:", os.environ.get("MLFLOW_TRACKING_URI"))
print("MLFLOW_EXPERIMENT:", os.environ.get("MLFLOW_EXPERIMENT"))
print("CLASSIFICATION_THRESHOLD:", os.environ.get("CLASSIFICATION_THRESHOLD"))

HF_DATASET_REPO: abhisekbasu/tourism-project-data
HF_MODEL_REPO: abhisekbasu/tourism-project-model
TARGET_COL: ProdTaken
MLFLOW_TRACKING_URI: http://localhost:5000
MLFLOW_EXPERIMENT: MLOps_experiment
CLASSIFICATION_THRESHOLD: 0.45


In [29]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/tourism_project/model_building/train.py
# for data manipulation
import os
import json
import pandas as pd

# for data preprocessing and pipeline creation
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# for model training, tuning, and evaluation
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# for model serialization
import joblib

# for hugging face model hub upload
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

# for experimentation tracking
import mlflow


def main():
    # -------------------------
    # ENV VARS (set in Colab now; later in GitHub Actions)
    # -------------------------
    HF_DATASET_REPO = os.environ.get("HF_DATASET_REPO")     # abhisekbasu/tourism-project-data
    HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO")         # abhisekbasu/tourism-project-model
    TARGET_COL = os.environ.get("TARGET_COL", "ProdTaken")

    TRAIN_PATH_IN_REPO = os.environ.get("TRAIN_PATH_IN_REPO", "processed/train.csv")
    TEST_PATH_IN_REPO  = os.environ.get("TEST_PATH_IN_REPO",  "processed/test.csv")

    MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
    MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", "MLOps_experiment")

    CLASSIFICATION_THRESHOLD = float(os.environ.get("CLASSIFICATION_THRESHOLD", "0.45"))

    if not HF_DATASET_REPO:
        raise ValueError("Missing HF_DATASET_REPO (example: 'abhisekbasu/tourism-project-data')")
    if not HF_MODEL_REPO:
        raise ValueError("Missing HF_MODEL_REPO (example: 'abhisekbasu/tourism-project-model')")

    # -------------------------
    # MLflow config (env-driven)
    # -------------------------
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    # -------------------------
    # Load train/test from HF Dataset repo (hf:// style like sample)
    # -------------------------
    Xtrain_path = f"hf://datasets/{HF_DATASET_REPO}/{TRAIN_PATH_IN_REPO}"
    Xtest_path  = f"hf://datasets/{HF_DATASET_REPO}/{TEST_PATH_IN_REPO}"

    train_df = pd.read_csv(Xtrain_path)
    test_df  = pd.read_csv(Xtest_path)

    if TARGET_COL not in train_df.columns:
        raise ValueError(f"Target '{TARGET_COL}' not found. Columns: {list(train_df.columns)}")

    Xtrain = train_df.drop(columns=[TARGET_COL])
    ytrain = train_df[TARGET_COL].astype(int)

    Xtest = test_df.drop(columns=[TARGET_COL])
    ytest = test_df[TARGET_COL].astype(int)

    # -------------------------
    # One-hot encode categorical + scale numeric (explicit lists like sample)
    # -------------------------
    numeric_features = [
        "Age",
        "CityTier",
        "DurationOfPitch",
        "NumberOfPersonVisiting",
        "NumberOfFollowups",
        "PreferredPropertyStar",
        "NumberOfTrips",
        "Passport",
        "PitchSatisfactionScore",
        "OwnCar",
        "NumberOfChildrenVisiting",
        "MonthlyIncome",
    ]
    categorical_features = [
        "TypeofContact",
        "Occupation",
        "Gender",
        "ProductPitched",
        "MaritalStatus",
        "Designation",
    ]

    # Optional safety checks (helps avoid silent failures in CI/CD)
    missing_num = [c for c in numeric_features if c not in Xtrain.columns]
    missing_cat = [c for c in categorical_features if c not in Xtrain.columns]
    if missing_num or missing_cat:
        raise ValueError(f"Feature mismatch. Missing numeric={missing_num}, missing categorical={missing_cat}")

    preprocessor = make_column_transformer(
        (StandardScaler(), numeric_features),
        (OneHotEncoder(handle_unknown="ignore"), categorical_features)
    )

    # -------------------------
    # Handle class imbalance (same idea as sample)
    # -------------------------
    class_weight = ytrain.value_counts()[0] / ytrain.value_counts()[1]

    # -------------------------
    # Define base XGBoost model + hyperparameter grid (sample style)
    # -------------------------
    xgb_model = xgb.XGBClassifier(
        scale_pos_weight=class_weight,
        random_state=42,
        eval_metric="logloss"
    )

    param_grid = {
        "xgbclassifier__n_estimators": [80, 120],
        "xgbclassifier__max_depth": [3, 4],
        "xgbclassifier__colsample_bytree": [0.5, 0.7],
        "xgbclassifier__learning_rate": [0.05, 0.1],
        "xgbclassifier__reg_lambda": [0.5, 1.0],
    }

    model_pipeline = make_pipeline(preprocessor, xgb_model)

    # -------------------------
    # Train + log ALL tuned parameters (nested runs like sample)
    # -------------------------
    with mlflow.start_run(run_name="tourism_prod_gridsearch"):
        grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, n_jobs=-1)
        grid_search.fit(Xtrain, ytrain)

        results = grid_search.cv_results_
        for i in range(len(results["params"])):
            param_set = results["params"][i]
            mean_score = results["mean_test_score"][i]
            std_score = results["std_test_score"][i]

            with mlflow.start_run(nested=True):
                mlflow.log_params(param_set)
                mlflow.log_metric("mean_test_score", float(mean_score))
                mlflow.log_metric("std_test_score", float(std_score))

        # Best params in parent run
        mlflow.log_params(grid_search.best_params_)

        best_model = grid_search.best_estimator_

        # Evaluate best model with fixed threshold
        y_pred_train_proba = best_model.predict_proba(Xtrain)[:, 1]
        y_pred_train = (y_pred_train_proba >= CLASSIFICATION_THRESHOLD).astype(int)

        y_pred_test_proba = best_model.predict_proba(Xtest)[:, 1]
        y_pred_test = (y_pred_test_proba >= CLASSIFICATION_THRESHOLD).astype(int)

        train_report = classification_report(ytrain, y_pred_train, output_dict=True)
        test_report = classification_report(ytest, y_pred_test, output_dict=True)

        mlflow.log_metrics({
            "train_accuracy": train_report["accuracy"],
            "train_precision": train_report["1"]["precision"],
            "train_recall": train_report["1"]["recall"],
            "train_f1": train_report["1"]["f1-score"],
            "test_accuracy": test_report["accuracy"],
            "test_precision": test_report["1"]["precision"],
            "test_recall": test_report["1"]["recall"],
            "test_f1": test_report["1"]["f1-score"],
        })

        # -------------------------
        # Save model locally + log artifacts
        # -------------------------
        os.makedirs("artifacts", exist_ok=True)

        model_path = "artifacts/best_tourism_model_v1.joblib"
        joblib.dump(best_model, model_path)

        summary = {
            "hf_dataset_repo": HF_DATASET_REPO,
            "hf_model_repo": HF_MODEL_REPO,
            "target_col": TARGET_COL,
            "classification_threshold": CLASSIFICATION_THRESHOLD,
            "best_params": grid_search.best_params_,
            "metrics": {
                "train_accuracy": train_report["accuracy"],
                "train_precision": train_report["1"]["precision"],
                "train_recall": train_report["1"]["recall"],
                "train_f1": train_report["1"]["f1-score"],
                "test_accuracy": test_report["accuracy"],
                "test_precision": test_report["1"]["precision"],
                "test_recall": test_report["1"]["recall"],
                "test_f1": test_report["1"]["f1-score"],
            }
        }
        summary_path = "artifacts/model_summary.json"
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)

        mlflow.log_artifact(model_path, artifact_path="model")
        mlflow.log_artifact(summary_path, artifact_path="model")

        # -------------------------
        # Upload best model to HF Model Hub
        # -------------------------
        api = HfApi()
        try:
            api.repo_info(repo_id=HF_MODEL_REPO, repo_type="model")
        except RepositoryNotFoundError:
            create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=False)

        api.upload_file(
            path_or_fileobj=model_path,
            path_in_repo="best_tourism_model_v1.joblib",
            repo_id=HF_MODEL_REPO,
            repo_type="model",
        )
        api.upload_file(
            path_or_fileobj=summary_path,
            path_in_repo="model_summary.json",
            repo_id=HF_MODEL_REPO,
            repo_type="model",
        )

        print("✅ Best model uploaded to HF Model Hub:", HF_MODEL_REPO)


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/Colab_Notebooks/tourism_project/model_building/train.py


# Deployment

## Dockerfile

In [7]:
os.makedirs("tourism_project/deployment", exist_ok=True)

In [8]:
import os
DEPLOY_DIR = "/content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment"
print(os.listdir(DEPLOY_DIR))

['app.py', 'requirements.txt']


In [9]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/Dockerfile
FROM python:3.10-slim

WORKDIR /app

# Minimal system deps
RUN apt-get update && apt-get install -y --no-install-recommends \
    git \
 && rm -rf /var/lib/apt/lists/*

COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py /app/app.py

# HF Spaces Docker expects service on 7860
EXPOSE 7860

CMD ["streamlit", "run", "app.py", "--server.port", "7860", "--server.address", "0.0.0.0"]

Writing /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/Dockerfile


In [38]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/Dockerfile
FROM python:3.12-slim

WORKDIR /app

# Minimal system deps
RUN apt-get update && apt-get install -y --no-install-recommends \
    git \
 && rm -rf /var/lib/apt/lists/*

COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py /app/app.py

EXPOSE 7860

CMD ["streamlit", "run", "app.py", "--server.port", "7860", "--server.address", "0.0.0.0"]

Overwriting /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/Dockerfile


In [10]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/push_to_hf.py
import os
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

def main():
    hf_space_repo = os.environ.get("HF_SPACE_REPO")
    if not hf_space_repo:
        raise ValueError("Missing HF_SPACE_REPO (example: 'abhisekbasu/tourism-project-app')")

    api = HfApi()

    # Create Docker Space if missing
    try:
        api.repo_info(repo_id=hf_space_repo, repo_type="space")
        print(f"✅ Space exists: {hf_space_repo}")
    except RepositoryNotFoundError:
        create_repo(
            repo_id=hf_space_repo,
            repo_type="space",
            space_sdk="docker",
            private=False
        )
        print(f"✅ Created Docker Space: {hf_space_repo}")

    # Upload the deployment folder to the Space
    folder_path = os.path.dirname(os.path.abspath(__file__))
    api.upload_folder(
        repo_id=hf_space_repo,
        repo_type="space",
        folder_path=folder_path
    )

    print(f"✅ Uploaded deployment files to Space: {hf_space_repo}")

if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/push_to_hf.py


In [12]:
from huggingface_hub import notebook_login
notebook_login()

In [13]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(repo_id="abhisekbasu/tourism-project-model", repo_type="model")
print(files)

RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-69a3bc4d-435721db23f015c72eb55fca;cef9ccc4-e230-499c-bbea-e6fa94f06466)

Repository Not Found for url: https://huggingface.co/api/models/abhisekbasu/tourism-project-model/tree/main?recursive=true&expand=false.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication

In [14]:
from huggingface_hub import notebook_login
notebook_login()

In [15]:
from huggingface_hub import HfApi
api = HfApi()
print("✅ Logged in as:", api.whoami()["name"])

✅ Logged in as: abhisekbasu


In [16]:
from huggingface_hub import HfApi, create_repo

repo_id = "abhisekbasu/tourism-project-model"

# Creates if not exists; if exists it won't break
create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)

api = HfApi()
print("✅ Model repo ready:", repo_id)
print(api.repo_info(repo_id=repo_id, repo_type="model"))

✅ Model repo ready: abhisekbasu/tourism-project-model
ModelInfo(id='abhisekbasu/tourism-project-model', author='abhisekbasu', sha='b29764718bdbead0ac79ee043d7b79615eb77664', created_at=datetime.datetime(2026, 3, 1, 4, 13, 8, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2026, 3, 1, 4, 13, 8, tzinfo=datetime.timezone.utc), private=False, disabled=False, downloads=0, downloads_all_time=None, gated=False, gguf=None, inference=None, inference_provider_mapping=None, likes=0, library_name=None, tags=['region:us'], pipeline_tag=None, mask_token=None, card_data=None, widget_data=None, model_index=None, config={}, transformers_info=None, trending_score=None, siblings=[RepoSibling(rfilename='.gitattributes', size=None, blob_id=None, lfs=None)], spaces=[], safetensors=None, security_repo_status=None, eval_results=None)


In [17]:
from huggingface_hub import HfApi
api = HfApi()
print(api.list_repo_files(repo_id="abhisekbasu/tourism-project-model", repo_type="model"))

['.gitattributes']


In [18]:
from huggingface_hub import HfApi, create_repo

dataset_repo = "abhisekbasu/tourism-project-data"
create_repo(repo_id=dataset_repo, repo_type="dataset", private=False, exist_ok=True)

api = HfApi()
print("✅ Dataset repo ready:", dataset_repo)

✅ Dataset repo ready: abhisekbasu/tourism-project-data


In [19]:
import os

os.environ["HF_DATASET_REPO"] = "abhisekbasu/tourism-project-data"
os.environ["HF_MODEL_REPO"] = "abhisekbasu/tourism-project-model"
os.environ["TARGET_COL"] = "ProdTaken"

# these paths are what train.py expects
os.environ["TRAIN_PATH_IN_REPO"] = "processed/train.csv"
os.environ["TEST_PATH_IN_REPO"] = "processed/test.csv"

print("✅ Env set for dataset + model repos")

✅ Env set for dataset + model repos


In [20]:
local_csv_path = os.environ.get("LOCAL_CSV_PATH")

In [21]:
import os

os.environ["HF_DATASET_REPO"] = "abhisekbasu/tourism-project-data"
os.environ["LOCAL_CSV_PATH"] = "/content/drive/MyDrive/Colab_Notebooks/tourism_project/data/tourism.csv"
os.environ["TARGET_COL"] = "ProdTaken"

print("HF_DATASET_REPO:", os.environ["HF_DATASET_REPO"])
print("LOCAL_CSV_PATH exists?", os.path.exists(os.environ["LOCAL_CSV_PATH"]))

HF_DATASET_REPO: abhisekbasu/tourism-project-data
LOCAL_CSV_PATH exists? True


In [22]:
!python /content/drive/MyDrive/Colab_Notebooks/tourism_project/model_building/prep.py

Uploaded raw data to HF: abhisekbasu/tourism-project-data/raw/tourism.csv
Saved locally: data/processed/train.csv data/processed/test.csv
Uploaded processed splits to HF: abhisekbasu/tourism-project-data/processed/train.csv and abhisekbasu/tourism-project-data/processed/test.csv


In [23]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(repo_id="abhisekbasu/tourism-project-data", repo_type="dataset")
print(files)

['.gitattributes', 'processed/test.csv', 'processed/train.csv', 'raw/tourism.csv']


In [24]:
import os

os.environ["HF_DATASET_REPO"] = "abhisekbasu/tourism-project-data"
os.environ["HF_MODEL_REPO"] = "abhisekbasu/tourism-project-model"
os.environ["TARGET_COL"] = "ProdTaken"

# train.py expects these HF paths
os.environ["TRAIN_PATH_IN_REPO"] = "processed/train.csv"
os.environ["TEST_PATH_IN_REPO"] = "processed/test.csv"

# MLflow (keep same as your dev setup; works if mlflow ui is still running)
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT"] = "MLOps_experiment"
os.environ["CLASSIFICATION_THRESHOLD"] = "0.45"

print("✅ Env set for train.py")

✅ Env set for train.py


In [29]:
import os

# File-based MLflow tracking store (works without any server)
os.environ["MLFLOW_TRACKING_URI"] = "file:/content/mlruns"
os.environ["MLFLOW_EXPERIMENT"] = "MLOps_experiment"

print("MLFLOW_TRACKING_URI =", os.environ["MLFLOW_TRACKING_URI"])

MLFLOW_TRACKING_URI = file:/content/mlruns


In [30]:
!python /content/drive/MyDrive/Colab_Notebooks/tourism_project/model_building/train.py

2026/03/01 04:35:38 INFO mlflow.tracking.fluent: Experiment with name 'MLOps_experiment' does not exist. Creating a new experiment.
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...t_tourism_model_v1.joblib: 100% 205k/205k [00:00<?, ?B/s]

Processing Files (1 / 1)      : 100% 205k/205k [00:00<00:00, 394kB/s, 1.02MB/s  ]
New Data Upload               : 100% 205k/205k [00:00<00:00, 394kB/s, 1.02MB/s  ]

  ...t_tourism_model_v1.joblib: 100% 205k/205k [00:00<?, ?B/s]

Processing Files (1 / 1)      : 100% 205k/205k [00:00<00:00, 284kB/s,  512kB/s  ]
New Data Upload               : 100% 205k/205k [00:00<00:00, 284kB/s,  512kB/s  ]
  ...t_tourism_model_v1.joblib: 100% 205k/205k [00:00<?, ?B/s]
✅ Best model uploaded to HF Model Hub: abhisekbasu/tourism-project-model


In [31]:
from huggingface_hub import HfApi
api = HfApi()
print(api.list_repo_files(repo_id="abhisekbasu/tourism-project-model", repo_type="model"))

['.gitattributes', 'best_tourism_model_v1.joblib', 'model_summary.json']


In [32]:
import os

os.environ["HF_SPACE_REPO"] = "abhisekbasu/tourism-project-app"

# app.py reads these to locate the model
os.environ["HF_MODEL_REPO"] = "abhisekbasu/tourism-project-model"
os.environ["HF_MODEL_FILENAME"] = "best_tourism_model_v1.joblib"

print("✅ Env set for Space push")
print("HF_SPACE_REPO:", os.environ["HF_SPACE_REPO"])

✅ Env set for Space push
HF_SPACE_REPO: abhisekbasu/tourism-project-app


In [33]:
!python /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/push_to_hf.py

✅ Created Docker Space: abhisekbasu/tourism-project-app
✅ Uploaded deployment files to Space: abhisekbasu/tourism-project-app


In [36]:
import sys, sklearn, joblib, xgboost
print("python:", sys.version)
print("sklearn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("xgboost:", xgboost.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
sklearn: 1.6.1
joblib: 1.5.3
xgboost: 3.2.0


In [35]:
from huggingface_hub import HfApi
api = HfApi()

info = api.repo_info(repo_id="abhisekbasu/tourism-project-app", repo_type="space")

print("✅ Space exists:", info.id)
print("Type:", info.type)
print("SDK:", getattr(info, "sdk", "N/A"))

✅ Space exists: abhisekbasu/tourism-project-app


AttributeError: 'SpaceInfo' object has no attribute 'type'

In [39]:
import os
os.environ["HF_SPACE_REPO"] = "abhisekbasu/tourism-project-app"
!python /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/push_to_hf.py

✅ Space exists: abhisekbasu/tourism-project-app
✅ Uploaded deployment files to Space: abhisekbasu/tourism-project-app


## Streamlit App

Please ensure that the web app script is named `app.py`.

In [30]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/app.py
import os
import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

st.set_page_config(page_title="Tourism Package Prediction", layout="centered")

HF_MODEL_REPO = os.environ.get("HF_MODEL_REPO", "abhisekbasu/tourism-project-model")
MODEL_FILENAME = os.environ.get("HF_MODEL_FILENAME", "best_tourism_model_v1.joblib")

@st.cache_resource
def load_model():
    model_path = hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename=MODEL_FILENAME,
        repo_type="model"
    )
    return joblib.load(model_path)

model = load_model()

st.title("Tourism Package Prediction")
st.write("Predict whether a customer will purchase the Wellness Tourism Package (ProdTaken: 0/1).")

# Input fields (match training columns)
age = st.number_input("Age", min_value=0.0, max_value=120.0, value=35.0, step=1.0)
typeofcontact = st.selectbox("TypeofContact", ["Company Invited", "Self Inquiry"])
citytier = st.selectbox("CityTier", [1, 2, 3])
durationofpitch = st.number_input("DurationOfPitch", min_value=0.0, max_value=300.0, value=15.0, step=1.0)
occupation = st.selectbox("Occupation", ["Salaried", "Small Business", "Free Lancer", "Other"])
gender = st.selectbox("Gender", ["Male", "Female"])
num_person = st.number_input("NumberOfPersonVisiting", min_value=1, max_value=20, value=2, step=1)
num_followups = st.number_input("NumberOfFollowups", min_value=0.0, max_value=20.0, value=3.0, step=1.0)
productpitched = st.selectbox("ProductPitched", ["Basic", "Deluxe", "Standard", "Super Deluxe", "King", "Other"])
preferred_star = st.number_input("PreferredPropertyStar", min_value=1.0, max_value=5.0, value=3.0, step=1.0)
marital = st.selectbox("MaritalStatus", ["Single", "Married", "Divorced", "Unmarried", "Other"])
num_trips = st.number_input("NumberOfTrips", min_value=0.0, max_value=50.0, value=2.0, step=1.0)
passport = st.selectbox("Passport", [0, 1])
pitch_score = st.selectbox("PitchSatisfactionScore", [1, 2, 3, 4, 5])
owncar = st.selectbox("OwnCar", [0, 1])
num_children = st.number_input("NumberOfChildrenVisiting", min_value=0.0, max_value=10.0, value=0.0, step=1.0)
designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP", "Other"])
monthly_income = st.number_input("MonthlyIncome", min_value=0.0, max_value=10000000.0, value=30000.0, step=1000.0)

input_df = pd.DataFrame([{
    "Age": age,
    "TypeofContact": typeofcontact,
    "CityTier": int(citytier),
    "DurationOfPitch": durationofpitch,
    "Occupation": occupation,
    "Gender": gender,
    "NumberOfPersonVisiting": int(num_person),
    "NumberOfFollowups": num_followups,
    "ProductPitched": productpitched,
    "PreferredPropertyStar": preferred_star,
    "MaritalStatus": marital,
    "NumberOfTrips": num_trips,
    "Passport": int(passport),
    "PitchSatisfactionScore": int(pitch_score),
    "OwnCar": int(owncar),
    "NumberOfChildrenVisiting": num_children,
    "Designation": designation,
    "MonthlyIncome": monthly_income
}])

if st.button("Predict"):
    pred = model.predict(input_df)[0]
    proba = None
    if hasattr(model, "predict_proba"):
        proba = float(model.predict_proba(input_df)[:, 1][0])

    if int(pred) == 1:
        st.success(f"Prediction: WILL purchase (ProdTaken = 1)")
    else:
        st.warning(f"Prediction: will NOT purchase (ProdTaken = 0)")

    if proba is not None:
        st.info(f"Purchase probability (class 1): {proba:.3f}")

st.caption(f"Model source: {HF_MODEL_REPO}/{MODEL_FILENAME}")

Writing /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/app.py


## Dependency Handling

Please ensure that the dependency handling file is named `requirements.txt`.

In [37]:
%%writefile /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/requirements.txt
streamlit==1.33.0
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.6.1
xgboost==3.2.0
joblib==1.5.3
huggingface_hub==0.24.6

Overwriting /content/drive/MyDrive/Colab_Notebooks/tourism_project/deployment/requirements.txt


# Hosting

In [40]:
import os

GITHUB_USER = "abhisekbasu"
REPO_NAME = "tourism-project-mlops"
RUNTIME_REPO_DIR = f"/content/{REPO_NAME}"

# Clean old clone if exists
if os.path.exists(RUNTIME_REPO_DIR):
    !rm -rf {RUNTIME_REPO_DIR}

# Clone
!git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {RUNTIME_REPO_DIR}

print("✅ Cloned:", RUNTIME_REPO_DIR)
!ls -la {RUNTIME_REPO_DIR}

Cloning into '/content/tourism-project-mlops'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
✅ Cloned: /content/tourism-project-mlops
total 16
drwxr-xr-x 3 root root 4096 Mar  1 05:08 .
drwxr-xr-x 1 root root 4096 Mar  1 05:08 ..
drwxr-xr-x 8 root root 4096 Mar  1 05:08 .git
-rw-r--r-- 1 root root   23 Mar  1 05:08 README.md


In [ ]:
RUNTIME_REPO_DIR = "/content/tourism-project-mlops"

!git -C {RUNTIME_REPO_DIR} config user.name "abhisekbasu"
!git -C {RUNTIME_REPO_DIR} config user.email "abhisekbasu@users.noreply.github.com"
print("✅ Git identity set")

# MLOps Pipeline with Github Actions Workflow

**Note:**

1. Before running the file below, make sure to add the HF_TOKEN to your GitHub secrets to enable authentication between GitHub and Hugging Face.
2. The below code is for a sample YAML file that can be updated as required to meet the requirements of this project.

```
name: Tourism Project Pipeline

on:
  push:
    branches:
      - main  # Automatically triggers on push to the main branch

jobs:

  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Upload Dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Run Data Preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  model-traning:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Start MLflow Server
        run: |
          nohup mlflow ui --host 0.0.0.0 --port 5000 &  # Run MLflow UI in the background
          sleep 5  # Wait for a moment to let the server starts
      - name: Model Building
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>


  deploy-hosting:
    runs-on: ubuntu-latest
    needs: [model-traning,data-prep,register-dataset]
    steps:
      - uses: actions/checkout@v3
      - name: Install Dependencies
        run: <add_code_here>
      - name: Push files to Frontend Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
        run: <add_code_here>

```

**Note:** To use this YAML file for our use case, we need to

1. Go to the GitHub repository for the project
2. Create a folder named ***.github/workflows/***
3. In the above folder, create a file named ***pipeline.yml***
4. Copy and paste the above content for the YAML file into the ***pipeline.yml*** file

## Requirements file for the Github Actions Workflow

## Github Authentication and Push Files

* Before moving forward, we need to generate a secret token to push files directly from Colab to the GitHub repository.
* Please follow the below instructions to create the GitHub token:
    - Open your GitHub profile.
    - Click on ***Settings***.
    - Go to ***Developer Settings***.
    - Expand the ***Personal access tokens*** section and select ***Tokens (classic)***.
    - Click ***Generate new token***, then choose ***Generate new token (classic)***.
    - Add a note and select all required scopes.
    - Click ***Generate token***.
    - Copy the generated token and store it safely in a notepad.

In [ ]:
# Install Git
!apt-get install git

# Set your Git identity (replace with your details)
!git config --global user.email "<-------GitHub Email Address------->"
!git config --global user.name "<--------GitHub UserName--------->"

# Clone your GitHub repository
!git clone https://github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Move your folder to the repository directory
!mv /content/tourism_project/ /content/<--------GitHub Reponame--------->

In [ ]:
# Change directory to the cloned repository
%cd <--------GitHub Reponame--------->/

# Add the new folder to Git
!git add .

# Commit the changes
!git commit -m "first commit"

# Push to GitHub (you'll need your GitHub credentials; use a personal access token if 2FA enabled)
!git push https://<--------GitHub UserName--------->:<--------GitHub Token--------->@github.com/<--------GitHub UserName--------->/<--------GitHub Reponame--------->.git

# Output Evaluation

- GitHub (link to repository, screenshot of folder structure and executed workflow)

- Streamlit on Hugging Face (link to HF space, screenshot of Streamlit app)

<font size=6 color="navyblue">Power Ahead!</font>
___